# Multi-Modal RAG with Image Captioning

## Overview

Standard RAG only processes **text**. But many documents (research papers, reports) contain **images, figures, and tables** that hold critical information. Multi-modal RAG extracts and processes both.

| Standard RAG | Multi-Modal RAG |
|---|---|
| Extract text only | Extract text **+ images** |
| Images are ignored | Images are **captioned by a vision model** |
| Vector store has text chunks | Vector store has text chunks **+ image captions** |

## Pipeline

1. Extract text and images from a PDF
2. Use a **vision model** to generate captions for each image
3. Chunk both text and image captions
4. Store everything in a single vector store
5. Query retrieves the most relevant chunk — whether it came from text or an image

<img src="./images/multi_model_rag_with_captioning.svg" alt="Multi-Modal RAG" width="300">

## Models Used

- **Vision LLM**: `gemma3:12b` via Ollama (image captioning)
- **Text LLM**: `gemma3:12b` via Ollama (answer generation)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama

---
## Step 0: Import Packages

In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os
import ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama

---
## Step 1: Download the Paper

We use the classic "Attention Is All You Need" paper as our test document — it has both text and important diagrams.

In [ ]:
if not os.path.exists("attention_is_all_you_need.pdf"):
    !wget -q https://arxiv.org/pdf/1706.03762 -O attention_is_all_you_need.pdf
    print("Downloaded paper")
else:
    print("Paper already exists")

---
## Step 2: Extract Text and Images from the PDF

We use PyMuPDF (`fitz`) to extract:
- **Text** from each page
- **Images** (figures, diagrams) saved to disk

In [ ]:
text_data = []
os.makedirs("extracted_images", exist_ok=True)
image_count = 0

with fitz.open("attention_is_all_you_need.pdf") as pdf_file:
    for page_number in range(len(pdf_file)):
        page = pdf_file[page_number]

        # Extract text
        text = page.get_text().strip()
        text_data.append({"response": text, "name": page_number + 1})

        # Extract images
        images = page.get_images(full=True)
        for image_index, img in enumerate(images):
            xref = img[0]
            base_image = pdf_file.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image = Image.open(io.BytesIO(image_bytes))
            image.save(f"extracted_images/image_{page_number+1}_{image_index+1}.{image_ext}")
            image_count += 1

print(f"Extracted text from {len(text_data)} pages")
print(f"Extracted {image_count} images")
print(f"Images saved to: extracted_images/")

---
## Step 3: Caption Each Image with a Vision Model

For each extracted image, we send it to `gemma3:12b` (which supports vision) and ask it to generate a concise summary/caption. These captions will be embedded and stored alongside the text.

In [ ]:
img_data = []

caption_prompt = (
    "You are an assistant tasked with summarizing tables, images and text for retrieval. "
    "These summaries will be embedded and used to retrieve the raw text or table elements. "
    "Give a concise summary of the table or text that is well optimized for retrieval. "
    "Table or text or image:"
)

image_files = sorted(os.listdir("extracted_images"))
for img_name in image_files:
    img_path = f"extracted_images/{img_name}"

    response = ollama.chat(
        model="gemma3:12b",
        messages=[{
            "role": "user",
            "content": caption_prompt,
            "images": [img_path],
        }],
    )

    caption = response.message.content
    img_data.append({"response": caption, "name": img_name})
    print(f"  {img_name}: {caption[:100]}...")

print(f"\nCaptioned {len(img_data)} images")

---
## Step 4: Chunk Text and Image Captions, Build Vector Store

We split both the page text and image captions into chunks, then store everything in a single Chroma vector store. At retrieval time, the query can match text chunks OR image caption chunks.

In [ ]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

docs_list = [Document(page_content=t["response"], metadata={"name": t["name"], "type": "text"}) for t in text_data]
img_list = [Document(page_content=i["response"], metadata={"name": i["name"], "type": "image"}) for i in img_data]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)
img_splits = text_splitter.split_documents(img_list)

print(f"Text chunks: {len(doc_splits)}")
print(f"Image caption chunks: {len(img_splits)}")

all_splits = doc_splits + img_splits
vectorstore = Chroma.from_documents(
    documents=all_splits,
    collection_name="multi_model_rag",
    embedding=embedding_model,
)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 1})

print(f"\nVector store created with {len(all_splits)} total chunks")
print(f"Retriever set to return top 1 result")

---
## Step 5: Query and Generate an Answer

We ask a question about a **figure** in the paper. The retriever should find the relevant image caption, and the LLM generates an answer from it.

In [ ]:
query = "how many boxes are in the Scaled Dot Product Attention?"
print(f"Query: {query}\n")

# Retrieve
docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} document(s):")
for d in docs:
    print(f"  Type: {d.metadata.get('type', 'unknown')}")
    print(f"  Source: {d.metadata.get('name', 'N/A')}")
    print(f"  Content: {d.page_content[:200]}...")

# Generate answer
llm = ChatOllama(model="gemma3:12b", temperature=0)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. Answer the question based upon your knowledge. "
     "Use three-to-five sentences maximum and keep the answer concise."),
    ("human",
     "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
])

answer_chain = answer_prompt | llm | StrOutputParser()
answer = answer_chain.invoke({"documents": docs[0].page_content, "question": query})

print(f"\nAnswer: {answer}")

---
## Summary

| Step | What happened |
|---|---|
| 1 | Downloaded the "Attention Is All You Need" paper |
| 2 | Extracted text (per page) and images from the PDF |
| 3 | **Captioned each image** using `gemma3:12b` vision model |
| 4 | Chunked both text and captions, stored in one Chroma vector store |
| 5 | Queried about a figure → retrieved the relevant image caption → generated answer |

**Key insight:** By captioning images and storing those captions as searchable text, we make visual content (diagrams, tables, figures) retrievable through the same vector search pipeline as regular text. A question about the "Scaled Dot Product Attention" diagram gets answered because the vision model's caption describes the diagram's contents.